# Notebook 04 — Merge TCGA-PAAD and 1000 Genomes Project

Combines `tcga_paad_alphamissense_inner.parquet` (somatic PDAC mutations, AM-annotated only)
with `1000G_alphamissense_inner.parquet` (germline missense variants, AM-annotated only) into a
single long-form parquet: `merged_tcga_1000G.parquet`. Both inputs use snake_case column names and have `uniprot_id` always non-null.

A `source` column (`'TCGA'` / `'1000G'`) marks each row's origin. `gene_symbol` (HGNC primary
symbol) is present for TCGA rows (from MAF) and added for 1000G rows via UniProt REST API lookup.
`gene_name` is then derived as `coalesce(gene_symbol, uniprot_id)` — giving every row a
non-null gene label for downstream grouping.

## Output schema — `merged_tcga_1000G.parquet`

One row per **mutation–sample pair** (TCGA) or **variant–sample pair** (1000G). `transcript_id` is present in both inputs but dropped at this step.

| Column | Type | TCGA | 1000G | Notes |
|--------|------|------|-------|-------|
| `source` | String | `'TCGA'` | `'1000G'` | Row origin |
| `chrom` | String | ✓ | ✓ | Chromosome (e.g. `chr1`) |
| `pos` | Int64 | ✓ | ✓ | GRCh38 position |
| `ref` | String | ✓ | ✓ | Reference allele |
| `alt` | String | ✓ | ✓ | Alternate allele |
| `sample_id` | String | renamed from `patient_barcode` | ✓ | TCGA 12-char barcode or 1000G individual ID |
| `gene_symbol` | String | from MAF `Hugo_Symbol`; ~30 nulls | via UniProt API; partial | HGNC primary symbol |
| `gene_name` | String | derived | derived | `coalesce(gene_symbol, uniprot_id)`; always non-null |
| `uniprot_id` | String | ✓ | ✓ | UniProt accession; always non-null (inner join inputs) |
| `protein_variant` | String | ✓ | ✓ | Amino acid substitution (e.g. `G12D`) |
| `am_pathogenicity` | Float64 | ✓ | ✓ | AlphaMissense score 0–1 |
| `am_class` | String | ✓ | ✓ | `likely_benign` / `ambiguous` / `likely_pathogenic` |
| `blosum62_score` | Int8 | derived | derived | BLOSUM62 substitution score for `protein_variant` (e.g. G→D = −1); null if no AM match |
| `af_1000g` | Float64 | from MAF VEP | from VCF INFO | 1000G allele frequency; null for TCGA rows not in 1000G |
| `dosage` | Int8 | null | ✓ | 1 = het, 2 = hom-alt |
| `subtype` | String | `classical`/`basal-like`/null | null | Bailey 2016 molecular subtype |
| `purity` | String | `high`/`low`/null | null | Bailey 2016 purity classification |

**Input files:**
- `tcga_paad_alphamissense_inner.parquet` — AM-annotated somatic mutations; `uniprot_id` always non-null
- `1000G_alphamissense_inner.parquet` — AM-annotated germline variants; `uniprot_id` always non-null
- `uniprot_hugo_map.parquet` — cached UniProt → HGNC symbol map for 1000G rows

In [ ]:
import os
import polars as pl
import requests
import time
from pathlib import Path
from Bio.Align import substitution_matrices

BASE_DIR   = Path(r'C:\Users\daane\Desktop\DATA')
TCGA_PATH  = BASE_DIR / 'TCGA-PAAD' / 'tcga_paad_alphamissense_inner.parquet'
G1000_PATH = BASE_DIR / '1000G'     / '1000G_alphamissense_inner.parquet'
HUGO_CACHE = BASE_DIR / 'uniprot_hugo_map.parquet'
OUT_PATH   = BASE_DIR / 'merged_tcga_1000G.parquet'

# Canonical column order for the merged output
FINAL_COLS = [
    "source", "chrom", "pos", "ref", "alt", "sample_id",
    "gene_symbol", "gene_name", "uniprot_id",
    "protein_variant", "am_pathogenicity", "am_class", "blosum62_score",
    "af_1000g", "dosage",
    "subtype", "purity",
]

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

---
## 1 — Load Data

Reads both inner-join parquets and prints their shape and column lists to verify the schema matches expectations before the merge.

**Expected TCGA columns (14):** `chrom, pos, ref, alt, patient_barcode, gene_symbol, uniprot_id, transcript_id, protein_variant, am_pathogenicity, am_class, af_1000g, subtype, purity`

**Expected 1000G columns (12):** `chrom, pos, ref, alt, sample_id, dosage, uniprot_id, transcript_id, protein_variant, am_pathogenicity, am_class, af_1000g`

Null counts in inputs: TCGA has `af_1000g` null for 15,171 rows and `subtype`/`purity` null for 726 rows; all other TCGA columns are complete. 1000G has no nulls.

In [ ]:
tcga  = pl.read_parquet(TCGA_PATH)
g1000 = pl.read_parquet(G1000_PATH)

print(f'TCGA  — shape: {tcga.shape}')
print(f'1000G — shape: {g1000.shape}')
print()
print('TCGA columns: ', tcga.columns)
print('1000G columns:', g1000.columns)

---
## 2 — Build `gene_symbol` Lookup via UniProt REST API

Collects all unique `uniprot_id` values from both datasets and queries the UniProt REST API
in batches of 50. Results are cached to `uniprot_hugo_map.parquet`; if the cache already exists
it is loaded directly to avoid repeated network calls.

The lookup resolves UniProt accessions to **primary HGNC gene symbols** (`gene_symbol`). For the
TCGA portion this is already present from the MAF; for the 1000G portion it is absent and must be
derived here. After the merge, `gene_name = coalesce(gene_symbol, uniprot_id)` is added so every
row has a non-null gene label.

**API endpoints used:**
- Batch: `GET /uniprotkb/search?query=accession:X OR accession:Y&fields=accession,gene_names&format=tsv` — up to 50 accessions per request
- Fallback: `GET /uniprotkb/{id}?format=json` — for individual IDs missed by the batch query

**Output**: `uniprot_hugo_map.parquet` — columns `uniprot_id` (String), `gene_symbol` (String); **18,540 entries**. Results are stable (UniProt primary symbols change rarely); refresh by deleting `uniprot_hugo_map.parquet`.

In [ ]:
def fetch_hugo_symbols(uniprot_ids: list, batch_size: int = 50, delay: float = 0.25) -> dict:
    """Map UniProt accessions to primary Hugo gene symbols via the UniProt search API.

    The `gene_names` field returns all names space-separated (primary first, then aliases).
    We take only the first token as the primary HGNC symbol.
    """
    ids = list(set(uniprot_ids))
    n   = len(ids)
    mapping = {}
    for i in range(0, n, batch_size):
        batch = ids[i:i + batch_size]
        query = ' OR '.join(f'accession:{uid}' for uid in batch)
        resp  = requests.get(
            'https://rest.uniprot.org/uniprotkb/search',
            params={'query': query, 'fields': 'accession,gene_names', 'format': 'tsv', 'size': 500},
            timeout=30,
        )
        resp.raise_for_status()
        for line in resp.text.splitlines()[1:]:          # row 0 is the TSV header
            acc, _, names = line.partition('\t')
            acc   = acc.strip()
            names = names.strip()
            if acc and names:
                mapping[acc] = names.split()[0]          # first token = primary symbol
        time.sleep(delay)
        if (i // batch_size) % 20 == 0:
            print(f'  {min(i + batch_size, n):,}/{n:,}', end='\r')
    print(f'  {n:,}/{n:,} — done')
    return mapping

In [ ]:
def probe_single(uid: str) -> str | None:
    """Query one UniProtKB entry directly; returns the primary Hugo gene symbol or None."""
    try:
        resp = requests.get(
            f'https://rest.uniprot.org/uniprotkb/{uid}',
            params={'format': 'json'},
            timeout=15,
        )
        if not resp.ok:
            return None
        genes = resp.json().get('genes', [])
        return next((g['geneName']['value'] for g in genes if 'geneName' in g), None)
    except Exception:
        return None

In [ ]:
if HUGO_CACHE.exists():
    print('Loading gene_symbol cache ...')
    hugo_map = pl.read_parquet(HUGO_CACHE)
else:
    print('Cache not found — querying UniProt API ...')
    all_ids = (
        pl.concat([
            tcga ['uniprot_id'].drop_nulls(),
            g1000['uniprot_id'].drop_nulls(),
        ])
        .unique()
        .to_list()
    )
    print(f'Unique UniProt IDs to look up: {len(all_ids):,}')

    mapping = fetch_hugo_symbols(all_ids)

    missing = [uid for uid in all_ids if uid not in mapping]
    if missing:
        print(f'{len(missing):,} IDs missed by batch fetch — probing individually ...')
        for uid in missing:
            gene = probe_single(uid)
            if gene:
                mapping[uid] = gene
            time.sleep(0.1)
        print(f'  resolved {sum(1 for uid in missing if uid in mapping):,}/{len(missing):,}')

    hugo_map = pl.DataFrame({
        'uniprot_id':  list(mapping.keys()),
        'gene_symbol': list(mapping.values()),
    })
    hugo_map.write_parquet(HUGO_CACHE)
    print(f'Cache saved: {len(hugo_map):,} entries → {HUGO_CACHE}')

print(f'gene_symbol map: {len(hugo_map):,} entries')

---
## 3 — Prepare TCGA

**Input** (14 cols): `chrom, pos, ref, alt, patient_barcode, gene_symbol, uniprot_id, transcript_id, protein_variant, am_pathogenicity, am_class, af_1000g, subtype, purity`

**Changes:**
- Rename `patient_barcode` → `sample_id`
- Add `source = 'TCGA'`
- Drop `transcript_id` (not needed in merged output)
- Add null-typed column for 1000G-only field: `dosage` (Int8)

**Output** (15 cols): `source, chrom, pos, ref, alt, sample_id, gene_symbol, uniprot_id, protein_variant, am_pathogenicity, am_class, af_1000g, dosage, subtype, purity`

15,695 rows × 15 cols. `gene_symbol` null: 0 (all TCGA mutations have a MAF `Hugo_Symbol`). `dosage` null: 15,695 (TCGA-only field). `af_1000g` null: 15,171. `subtype`/`purity` null: 726.

In [ ]:
tcga_prep = (
    tcga
    .rename({'patient_barcode': 'sample_id'})
    .with_columns(pl.lit('TCGA').alias('source'))
    .select([
        'source', 'chrom', 'pos', 'ref', 'alt', 'sample_id',
        'gene_symbol', 'uniprot_id', 'protein_variant',
        'am_pathogenicity', 'am_class',
        'af_1000g',
        pl.lit(None).cast(pl.Int8).alias('dosage'),
        'subtype', 'purity',
    ])
)

print(f'TCGA prepared: {tcga_prep.shape}')

---
## 4 — Prepare 1000G

**Input** (12 cols): `chrom, pos, ref, alt, sample_id, dosage, uniprot_id, transcript_id, protein_variant, am_pathogenicity, am_class, af_1000g`

**Changes:**
- Join `hugo_map` on `uniprot_id` to add `gene_symbol` (left join — HGNC symbol from UniProt API; null for unresolved IDs)
- Add `source = '1000G'`
- Drop `transcript_id`
- Add null-typed columns for TCGA-only fields: `subtype` (String), `purity` (String)

**Output** (15 cols): `source, chrom, pos, ref, alt, sample_id, gene_symbol, uniprot_id, protein_variant, am_pathogenicity, am_class, af_1000g, dosage, subtype, purity`

32,063,253 rows × 15 cols. `gene_symbol` resolved for 32,016,896 / 32,063,253 rows (99.86%); 46,357 unresolved UniProt accessions remain null. `subtype`/`purity` null: 32,063,253 (always null for 1000G rows).

In [ ]:
g1000_prep = (
    g1000
    .join(hugo_map, on='uniprot_id', how='left')
    # Rename af→af_1000g for old parquets; new parquets already have af_1000g
    .rename({'af': 'af_1000g'} if 'af' in g1000.columns else {})
    .with_columns(pl.lit('1000G').alias('source'))
    .select([
        'source', 'chrom', 'pos', 'ref', 'alt', 'sample_id',
        'gene_symbol', 'uniprot_id', 'protein_variant',
        'am_pathogenicity', 'am_class',
        'af_1000g', 'dosage',
        pl.lit(None).cast(pl.String).alias('subtype'),
        pl.lit(None).cast(pl.String).alias('purity'),
    ])
)

n_gene_filled = g1000_prep['gene_symbol'].is_not_null().sum()
print(f'1000G prepared: {g1000_prep.shape}')
print(f'gene_symbol resolved: {n_gene_filled:,} / {len(g1000_prep):,} rows ({n_gene_filled/len(g1000_prep)*100:.1f}%)')

---
## 5 — Merge and Add `gene_name`

Concatenates both prepared DataFrames, then derives `gene_name` via `coalesce(gene_symbol, uniprot_id)`.

`uniprot_id` is always non-null in both inputs (inner join outputs), so `gene_name` is always non-null after coalesce — even for the small fraction of rows where `gene_symbol` is null.

**Output** (16 cols): all 15 prep columns + `gene_name`. **32,078,948 rows** (15,695 TCGA + 32,063,253 1000G). `gene_name` null: 0.

In [ ]:
merged = (
    pl.concat([tcga_prep, g1000_prep])
    .with_columns(
        pl.coalesce(['gene_symbol', 'uniprot_id']).alias('gene_name')
    )
)

print(f'Merged: {merged.shape}')

### 5b — Add `blosum62_score`

Adds a `blosum62_score` column derived from `protein_variant` (e.g. `G12D` → Gly→Asp) using the BLOSUM62 amino acid substitution matrix (Henikoff & Henikoff, 1992). BLOSUM62 is a log-odds scoring matrix derived from ungapped local alignments of evolutionarily diverged protein sequences; positive scores indicate conservative substitutions, negative scores indicate disruptive ones (e.g. Gly→Asp = −1).

`protein_variant` format: `{original_aa}{position}{new_aa}` (single-letter AA codes). The score is looked up via a vectorised join; null if `protein_variant` is null or contains a non-standard character.

The final `.select(FINAL_COLS)` reorders to canonical column order and drops intermediate columns.

**Output** (17 cols, 32,078,948 rows): all rows scored (100%), 0 nulls. TCGA subtype distribution: classical 13,428 · basal-like 1,541 · null 726.

In [ ]:
_mat = substitution_matrices.load("BLOSUM62")
_aas = [a for a in _mat.alphabet if a in set("ACDEFGHIKLMNPQRSTVWY")]
_blosum62 = pl.DataFrame(
    [(a, b, int(_mat[a][b])) for a in _aas for b in _aas],
    schema={"aa_orig": pl.String, "aa_new": pl.String, "blosum62_score": pl.Int8},
    orient="row",
)

merged = (
    merged
    .with_columns([
        pl.col("protein_variant").str.slice(0, 1).alias("_aa_orig"),
        pl.col("protein_variant").str.slice(-1, 1).alias("_aa_new"),
    ])
    .join(_blosum62, left_on=["_aa_orig", "_aa_new"], right_on=["aa_orig", "aa_new"], how="left")
    .drop(["_aa_orig", "_aa_new"])
    .select(FINAL_COLS)
)

---
### Sanity Checks

Seven assertions verify: row-count preservation per source, source-column exclusivity (`dosage` TCGA-only, `subtype`/`purity` TCGA-only), invariant columns (`gene_name` and `uniprot_id` never null), BLOSUM62 dtype correctness and known score (KRAS G12D = −1), and KRAS G12D presence in TCGA rows.

In [ ]:
# Row counts match inputs
assert merged.filter(pl.col('source') == 'TCGA' ).shape[0] == len(tcga_prep)
assert merged.filter(pl.col('source') == '1000G').shape[0] == len(g1000_prep)

# No TCGA row has dosage; no 1000G row has subtype/purity
assert merged.filter(pl.col('source') == 'TCGA' )['dosage'].is_not_null().sum() == 0
assert merged.filter(pl.col('source') == '1000G')['subtype'].is_not_null().sum() == 0

# gene_name is never null (both inputs use inner join → uniprot_id always non-null)
assert merged['gene_name'].is_null().sum() == 0, 'gene_name has unexpected nulls'

# uniprot_id is never null (inner join inputs)
assert merged['uniprot_id'].is_null().sum() == 0, 'uniprot_id has unexpected nulls'

# BLOSUM62 dtype and KRAS G12D score (Gly→Asp = -1)
assert merged.schema["blosum62_score"] == pl.Int8, "blosum62_score should be Int8"
kras_scores = merged.filter(
    (pl.col('source') == 'TCGA') &
    (pl.col('protein_variant') == 'G12D')
)['blosum62_score'].drop_nulls()
assert (kras_scores == -1).all(), 'KRAS G12D blosum62_score should be -1 (Gly→Asp)'

# KRAS G12D present in TCGA portion
kras = merged.filter(
    (pl.col('source') == 'TCGA') &
    (pl.col('gene_symbol') == 'KRAS') &
    (pl.col('protein_variant') == 'G12D')
)
assert len(kras) > 0, 'KRAS G12D missing from merged TCGA rows'

print('All sanity checks passed.')

---
## 6 — Save

Writes the merged DataFrame to disk.

**Output**: `merged_tcga_1000G.parquet` — **32,078,948 rows × 17 columns · 45.6 MB**

In [ ]:
merged.write_parquet(OUT_PATH)
print(f'Saved to {OUT_PATH}')
print(f'File size: {os.path.getsize(OUT_PATH) / 1024**2:.1f} MB')

---
## Summary

| Step | Input | Output | Notes |
|------|-------|--------|-------|
| 1 — Load data | `tcga_paad_alphamissense_inner.parquet`, `1000G_alphamissense_inner.parquet` | in-memory `tcga` (15,695 × 14), `g1000` (32,063,253 × 12) | Both inputs have `uniprot_id` always non-null |
| 2 — gene_symbol lookup | unique `uniprot_id` from both inputs | `uniprot_hugo_map.parquet` (18,540 entries) | Cached; batched UniProt REST API; fallback probes individual IDs |
| 3 — Prepare TCGA | `tcga` (14 cols) | `tcga_prep` (15 cols) | Rename `patient_barcode`→`sample_id`; add `source='TCGA'`; drop `transcript_id`; add null `dosage` |
| 4 — Prepare 1000G | `g1000` (12 cols) + `hugo_map` | `g1000_prep` (15 cols) | Join gene symbols (99.86% resolved); add `source='1000G'`; drop `transcript_id`; add null `subtype`/`purity` |
| 5 — Merge + gene_name + BLOSUM62 | `tcga_prep`, `g1000_prep` | in-memory `merged` (17 cols, 32,078,948 rows) | Concat; derive `gene_name = coalesce(gene_symbol, uniprot_id)`; join BLOSUM62 scores |
| 6 — Save | `merged` | `merged_tcga_1000G.parquet` (32,078,948 × 17, 45.6 MB) | |